# Model Evaluation

Runs NER, tags, and summary on 2 fixed eval docs for every model in `MODELS`.
Output is displayed inline for qualitative grading. Results are saved per model under `data/results/<model>/`.

In [1]:
MODELS = [
    "gemma2:2b",
    'qwen2.5:7b',
    "qwen2.5:14b",
    'qwen3:8b',
    'qwen3:14b',
    'llama3.1:8b',
    'llama3.2:latest',
    "mistral:7b",
    "granite3.3:8b",
    "deepseek-r1:8b",
    # "glm-5.1:cloud", 
    "gemma4:e4b",
    "lfm2:24b",
    "translategemma:12b",
    "rnj-1:8b",

    "qwen2.5:32b",
    "olmo-3:7b",
    "qwen3-vl:30b",
    "granite4:7b-a1b-h",

    "solar:10.7b",
    "falcon:7b",
    "gemma3n:e4b",
    "phi4-reasoning:14b",
    "deepseek-r1:14b",
    "deepseek-r1:32b",
    "gemma3:12b",
    "phi4:14b",
    "dolphin3:8b",
    "olmo2:13b",

    


]


In [2]:
%load_ext autoreload
%autoreload 2

import json
import sys
import time
sys.path.insert(0, '../src')

from pathlib import Path

from dotenv import load_dotenv

from notemine.config import load_config
from notemine.frontmatter_io import load_note
from notemine.tasks import run_ner, run_summary, run_tags

load_dotenv('../.env')
config = load_config('../config.toml')
config['vault']['prompts_dir'] = '../' + config['vault']['prompts_dir'].lstrip('./')

EVAL_DOCS_DIR = Path('../data/model_eval_docs')
RESULTS_DIR = Path('../data/results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DOCS = sorted(p.name for p in EVAL_DOCS_DIR.glob('*.md'))
backend = config['run']['backend']
print(f'backend={backend}  models={MODELS}  docs={EVAL_DOCS}')

backend=ollama  models=['gemma2:2b', 'qwen2.5:7b', 'qwen2.5:14b', 'qwen3:8b', 'qwen3:14b', 'llama3.1:8b', 'llama3.2:latest', 'mistral:7b', 'granite3.3:8b', 'deepseek-r1:8b', 'gemma4:e4b', 'lfm2:24b', 'translategemma:12b', 'rnj-1:8b', 'qwen2.5:32b', 'olmo-3:7b', 'qwen3-vl:30b', 'granite4:7b-a1b-h', 'solar:10.7b', 'falcon:7b', 'gemma3n:e4b', 'phi4-reasoning:14b', 'deepseek-r1:14b', 'deepseek-r1:32b', 'gemma3:12b', 'phi4:14b', 'dolphin3:8b', 'olmo2:13b']  docs=['medium_climate_blog_paris_agreement_en.md', 'medium_klimaat_blog_parijs_akkoord_nl.md']


In [3]:
all_results = {}

for model in MODELS:
    model_safe = model.replace(':', '_')
    tmp_path = RESULTS_DIR / f'model_eval_{model_safe}.json'

    if tmp_path.exists():
        print(f'--- {model}: skipping (found {tmp_path.name})')
        all_results[model] = json.loads(tmp_path.read_text())
        continue

    config[backend]['model'] = model
    print(f'\n--- {model} ---')
    results = []
    for doc in EVAL_DOCS:
        post = load_note(str(EVAL_DOCS_DIR / doc))
        row = {'file': doc}
        for task, fn in [('ner', run_ner), ('tags', run_tags), ('summary', run_summary)]:
            t0 = time.perf_counter()
            try:
                row[task] = fn(backend, post.content, config)
            except Exception as e:
                row[task] = f'ERROR: {e}'

            row[f'{task}_s'] = round(time.perf_counter() - t0, 1)
        results.append(row)

    tmp_path.write_text(json.dumps(results, ensure_ascii=False, indent=2))
    print(f'  saved {tmp_path.name}')
    all_results[model] = results

print('\nAll models done')

--- gemma2:2b: skipping (found model_eval_gemma2_2b.json)
--- qwen2.5:7b: skipping (found model_eval_qwen2.5_7b.json)
--- qwen2.5:14b: skipping (found model_eval_qwen2.5_14b.json)
--- qwen3:8b: skipping (found model_eval_qwen3_8b.json)
--- qwen3:14b: skipping (found model_eval_qwen3_14b.json)
--- llama3.1:8b: skipping (found model_eval_llama3.1_8b.json)
--- llama3.2:latest: skipping (found model_eval_llama3.2_latest.json)
--- mistral:7b: skipping (found model_eval_mistral_7b.json)
--- granite3.3:8b: skipping (found model_eval_granite3.3_8b.json)
--- deepseek-r1:8b: skipping (found model_eval_deepseek-r1_8b.json)
--- gemma4:e4b: skipping (found model_eval_gemma4_e4b.json)
--- lfm2:24b: skipping (found model_eval_lfm2_24b.json)
--- translategemma:12b: skipping (found model_eval_translategemma_12b.json)
--- rnj-1:8b: skipping (found model_eval_rnj-1_8b.json)
--- qwen2.5:32b: skipping (found model_eval_qwen2.5_32b.json)
--- olmo-3:7b: skipping (found model_eval_olmo-3_7b.json)
--- qwen3-vl

In [4]:
records = []
for model, results in all_results.items():
    for r in results:
        for task in ('ner', 'tags', 'summary'):
            payload = r[task]
            is_error = isinstance(payload, str) and payload.startswith('ERROR:')
            records.append({
                'model': model,
                'task': task,
                'doc': r['file'],
                'elapsed_s': r[f'{task}_s'],
                'result': payload if not is_error else None,
                'error': payload if is_error else None,
            })

out_path = RESULTS_DIR / 'model_eval.json'
out_path.write_text(json.dumps(records, ensure_ascii=False, indent=2))
print(f'Saved {len(records)} records to {out_path}')

Saved 168 records to ../data/results/model_eval.json
